# Wandas Pipeline Recipe UX

この Notebook は、Wandas の `RecipeSpec` と optional な sklearn 互換 transformer の UX を確認するための実行例です。

確認すること:

- Wandas-native な `RecipeSpec` で前処理を再実行可能にする。
- sklearn `Pipeline` の `transform(frame)` で同じ Wandas operation を適用する。
- `operation_history` で処理順とパラメータを確認する。
- 入力 frame を破壊せず、Dask-backed data の lazy graph を保つ。

## 1. Setup

sklearn adapter を使うため、環境には `wandas[sklearn]` が必要です。Wandas-native の `RecipeSpec` だけなら sklearn は不要です。

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from sklearn.pipeline import Pipeline

import wandas as wd
from wandas.pipeline import OperationSpec, RecipeSpec
from wandas.pipeline.sklearn import HighPassFilter, Normalize, RemoveDC

plt.rcParams["figure.figsize"] = (10, 4)

def channel_values(target_frame):
    return np.asarray(target_frame.data).reshape(-1)


print(f"Wandas version: {wd.__version__}")

## 2. Demo signal

50 Hz の低周波ドリフト、1000 Hz の主成分、3000 Hz の高周波成分、DC offset を含む合成信号を作ります。ファイルやネットワークには依存しません。

In [ ]:
sampling_rate = 16000
duration = 1.0
time = np.linspace(0, duration, int(sampling_rate * duration), endpoint=False)

raw_signal = (
    0.35
    + 0.7 * np.sin(2 * np.pi * 50 * time)
    + 1.0 * np.sin(2 * np.pi * 1000 * time)
    + 0.15 * np.sin(2 * np.pi * 3000 * time)
)

frame = wd.from_numpy(raw_signal.reshape(1, -1), sampling_rate=sampling_rate, ch_labels=["demo"])

print(f"sampling_rate: {frame.sampling_rate} Hz")
print(f"n_channels: {frame.n_channels}")
print(f"shape: {frame.data.shape}")
print(f"operation_history: {frame.operation_history}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(time[:800], channel_values(frame)[:800], label="raw")
ax.set_title("Raw signal preview")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.3)
ax.legend()
fig

## 3. Wandas-native RecipeSpec

`RecipeSpec` は Wandas の operation registry key とパラメータを順序付きで保持します。ここでは DC offset を除去し、100 Hz 未満を落とし、最後に正規化します。

In [ ]:
recipe = RecipeSpec(
    [
        OperationSpec("remove_dc"),
        OperationSpec("highpass_filter", {"cutoff": 100.0, "order": 2}),
        OperationSpec("normalize", {"norm": np.inf, "axis": -1, "threshold": None, "fill": None}),
    ]
)

recipe_processed = recipe.apply(frame)

print("Recipe steps:")
for step in recipe.steps:
    print(f"- {step.operation}: {dict(step.params)}")

print("\nResult operation history:")
print(json.dumps(recipe_processed.operation_history, indent=2, ensure_ascii=False))

print("\nOriginal frame still has no history:")
print(frame.operation_history)

In [ ]:
fig, ax = plt.subplots()
ax.plot(time[:800], channel_values(frame)[:800], label="raw", alpha=0.7)
ax.plot(time[:800], channel_values(recipe_processed)[:800], label="recipe processed", alpha=0.9)
ax.set_title("Before / after RecipeSpec")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.3)
ax.legend()
fig

## 4. sklearn Pipeline UX

sklearn に慣れているユーザーは、同じ Wandas operation を `Pipeline.transform(frame)` の形で実行できます。処理の正本は sklearn ではなく、結果 frame の `operation_history` です。

In [ ]:
sklearn_pipeline = Pipeline(
    [
        ("dc", RemoveDC()),
        ("hp", HighPassFilter(cutoff=100.0, order=2)),
        ("norm", Normalize(norm=np.inf, axis=-1, threshold=None, fill=None)),
    ]
)

sklearn_processed = sklearn_pipeline.transform(frame)

print(json.dumps(sklearn_processed.operation_history, indent=2, ensure_ascii=False))

recipe_ops = [record["operation"] for record in recipe_processed.operation_history]
sklearn_ops = [record["operation"] for record in sklearn_processed.operation_history]
print(f"\nSame operation order: {recipe_ops == sklearn_ops}")

## 5. Parameter editing and `to_spec()`

sklearn 風 transformer は `get_params()` / `set_params()` を持ちます。Notebook UI や探索コードでパラメータを変えたあと、`to_spec()` で Wandas-native な `OperationSpec` に戻せます。

In [ ]:
hp = HighPassFilter(cutoff=80.0)
print("Initial params:", hp.get_params())

hp.set_params(cutoff=120.0, order=6)
print("Edited params:", hp.get_params())

spec = hp.to_spec()
print("OperationSpec:", spec.operation, dict(spec.params))

## 6. Lazy execution check

`RecipeSpec.apply(frame)` は既存の `apply_operation` に委譲するだけなので、Dask-backed data は lazy graph のまま残ります。

In [ ]:
lazy_result = recipe.apply(frame)

print(f"input data type: {type(frame._data).__name__}")
print(f"result data type: {type(lazy_result._data).__name__}")
print(f"result chunks: {lazy_result._data.chunks}")
print(f"result dask graph tasks: {len(lazy_result._data.__dask_graph__())}")

## 7. Simple before / after summary

最後に、処理前後の平均値とピーク値を比較します。`remove_dc` と `highpass_filter` により DC/低周波成分が抑えられ、`normalize` によりピークが 1 に揃います。

In [ ]:
def summarize(label, target_frame):
    data = channel_values(target_frame)
    return {
        "label": label,
        "mean": float(np.mean(data)),
        "peak_abs": float(np.max(np.abs(data))),
        "history": [record["operation"] for record in target_frame.operation_history],
    }

for row in [summarize("raw", frame), summarize("recipe", recipe_processed), summarize("sklearn", sklearn_processed)]:
    print(json.dumps(row, ensure_ascii=False))

## Next steps

- WDF へ recipe を保存する場合は、別途 JSON schema と compatibility policy を設計します。
- sklearn `Pipeline` との自動相互変換は v1 では扱わず、まず `to_spec()` による片方向の明示変換に留めます。
- 複数入力 operation や FFT/STFT などの domain transition は、型と保存形式の設計後に拡張します。